In [0]:
%sql
SELECT * FROM bronze.sales

In [0]:
from pyspark.sql import functions as F, Window

TABLE = "workspace.bronze.sales"

source = spark.table(TABLE)
cols = source.columns
rows_before = source.count()

# ---------- 1. Detect exact duplicate rows (all columns identical) ----------
exact_dupes = source.groupBy(cols).count().filter(F.col("count") > 1)
n_exact_groups = exact_dupes.count()
n_exact_redundant = exact_dupes.agg(F.sum(F.col("count") - 1).alias("redundant")).collect()[0][0] or 0
print(f"Exact duplicate groups: {n_exact_groups} (redundant rows to remove: {n_exact_redundant})")

# ---------- 2. Detect conflicting order_ids (same order_id, >=1 differing column) ----------
# Remove exact duplicates first, so any remaining multiplicity on order_id means a real conflict
distinct_rows = source.dropDuplicates(cols)
conflicting = (
    distinct_rows
    .groupBy("order_id")
    .agg(F.count(F.lit(1)).alias("versions"))
    .filter(F.col("versions") > 1)
    .orderBy(F.col("versions").desc())
)
print(f"Conflicting order_ids (same order_id, differing column values): {conflicting.count()}")
conflicting.show(50, truncate=False)

# ---------- 3. Deduplicate: keep one of each exact duplicate, keep latest order_date per order_id ----------
# Rows with NULL order_id are kept as-is (no key to resolve conflicts on)
null_key = distinct_rows.filter(F.col("order_id").isNull())
nonnull_key = distinct_rows.filter(F.col("order_id").isNotNull())

w_latest = Window.partitionBy("order_id").orderBy(F.col("order_date").desc())
deduped = null_key.unionByName(
    nonnull_key
    .withColumn("_rn", F.row_number().over(w_latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

deduped.write.mode("overwrite").saveAsTable(TABLE)

rows_after = spark.table(TABLE).count()
print(f"Rows: {rows_before} -> {rows_after} (removed {rows_before - rows_after})")

In [0]:
from pyspark.sql import functions as F

TABLE = "workspace.bronze.sales"
NUM_COLS = ["unit_price", "discount", "revenue", "cost", "profit"]

source = spark.table(TABLE)
rows_before = source.count()

# ---------- Clean numeric columns: null/blank -> 0, round to 2 decimal places ----------
df = source
fixed_report = {}

for c in NUM_COLS:
    s = F.trim(F.col(c).cast("string"))
    is_blank = s.isNull() | (s == "") | s.isin("null", "NULL", "None", "NA", "N/A", "-")

    # count how many rows will be fixed (null/blank) in this column
    fixed_report[c] = source.filter(is_blank).count()

    df = df.withColumn(
        c,
        F.when(is_blank, F.lit(0.0))
         .otherwise(F.round(s.cast("double"), 2))
         .cast("decimal(18,2)")   # keeps exactly 2 decimal places (e.g. 10 -> 10.00)
    )

for c, n in fixed_report.items():
    print(f"{c}: {n} null/blank values replaced with 0")



In [0]:
df.display()

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.sales")

print(f"Wrote {spark.table('workspace.silver.sales').count()} rows to workspace.silver.sales")


In [0]:
%sql
select * from silver.customers